# Градиентный бустинг над деревьями

Учебный ноутбук по материалам семинара № 9 «Градиентный бустинг» (ФКН ВШЭ) и обсуждения функционального градиентного спуска.

**Цель:** понять, почему модель складывает деревья, что означает псевдоостаток, как выбор функции потерь меняет обучение и чем бустинг отличается от случайного леса.

## Содержание
1. [Идея и сумма моделей](#1-идея-и-сумма-моделей)
2. [Антиградиент и скалярное произведение](#2-антиградиент-и-скалярное-произведение)
3. [Потери и псевдоостатки](#3-потери-и-псевдоостатки)
4. [Алгоритм и пример на Python](#4-алгоритм-и-пример-на-python)
5. [Глубина деревьев и контроль переобучения](#5-глубина-деревьев-и-контроль-переобучения)
6. [Бустинг и случайный лес: сложность](#6-бустинг-и-случайный-лес-сложность)
7. [XGBoost, LightGBM, CatBoost](#7-xgboost-lightgbm-catboost)
8. [Как выбрать библиотеку и как устроены её деревья](#8-как-выбрать-библиотеку-и-как-устроены-её-деревья)

## 1. Идея и сумма моделей

Пусть есть пары $(x_i,y_i)$ и функция потерь $L(y,z)$, где $z$ — **числовой выход модели**. Минимизируем эмпирический риск:

$$Q(F)=\frac1n\sum_{i=1}^n L(y_i,F(x_i)).$$

Начинаем с простой константы $F_0(x)=\arg\min_c\sum_i L(y_i,c)$. Для квадратичной потери $L(y,z)=\tfrac12(y-z)^2$ это среднее $\bar y$.

На каждом шаге добавляем дерево как поправку:

$$F_m(x)=F_{m-1}(x)+\eta\gamma_m h_m(x),\qquad
F_M(x)=F_0(x)+\sum_{m=1}^M\eta\gamma_m h_m(x).$$

Здесь $\eta$ — темп обучения, а $\gamma_m$ — возможная оптимальная длина шага. Если она не подбирается отдельно, её эффект можно включить в значения листьев. Например, базовый прогноз 100 и поправки $+20,-5,+2$ дают 117. Дерево учится исправлять текущую композицию, поэтому возникает **сумма**.

## 2. Антиградиент и скалярное произведение

На обучающей выборке значения функции образуют вектор $\mathbf f=(F(x_1),\dots,F(x_n))$. Для $Q=\frac1n\sum_iL(y_i,f_i)$ производная по координате равна $\partial Q/\partial f_i=\frac1n\partial L(y_i,f_i)/\partial f_i$.

Если положить $g_i=\partial L(y_i,f_i)/\partial f_i$ и определить скалярное произведение $\langle u,v\rangle_n=\frac1n\sum_i u_iv_i$, то малое добавление $\varepsilon h$ даёт

$$Q(F+\varepsilon h)\approx Q(F)+\varepsilon\langle g,h\rangle_n.$$

Направление $h=-g$ уменьшает риск в первом приближении: $\langle g,-g\rangle_n=-\|g\|_n^2$. Поэтому цели для нового дерева — **псевдоостатки**

$$s_i=-\left.\frac{\partial L(y_i,z)}{\partial z}\right|_{z=F_{m-1}(x_i)}.$$

Множитель $1/n$ уже учтён в скалярном произведении: так не возникает противоречия между градиентом среднего риска и формулой для $s_i$. На конечной выборке хватает производных по $n$ числам. Производная Фреше нужна для строгой записи, когда аргумент риска — целая функция: $Q(F+h)=Q(F)+DQ[F](h)+o(\|h\|)$, причём в подходящем гильбертовом пространстве $DQ[F](h)=\langle\nabla Q(F),h\rangle$. Для вычисления алгоритма знать теорему Фреше не требуется.

**Оговорка:** направление самого быстрого спуска зависит от выбранной нормы и скалярного произведения. Дерево лишь приближает антиградиент, поэтому его добавление без подходящего шага не обязано уменьшать риск.

## 3. Потери и псевдоостатки

| Задача и потеря | $s=-\partial L/\partial z$ | Что означает |
|---|---|---|
| MSE, $\tfrac12(y-z)^2$ | $y-z$ | Величина обычной ошибки |
| MAE, $|y-z|$ | $\operatorname{sign}(y-z)$ | Направление исправления; в $z=y$ берут субградиент |
| Бинарная log-loss, $y\in\{0,1\}$, $L=-y\log p-(1-y)\log(1-p)$, $p=\sigma(z)$ | $y-p$ | Разность между меткой и вероятностью; $z$ — логит |

**Почему не всегда $y-z$?** Это антиградиент только для выбранной квадратичной потери (с указанным множителем $1/2$). При асимметричной потере из семинара завышение на 5 штрафуется в десять раз сильнее занижения на 5; антиградиенты равны $-50$ и $+5$. Обычные остатки $-5$ и $+5$ игнорировали бы этот приоритет.

MSE подходит, когда большие ошибки нужно штрафовать сильнее. При выбросах рассматривают Huber или MAE; для квантилей — квантильную потерю; для бинарной классификации — log-loss. Если функция недифференцируема, формула обычного градиента не действует в точках излома: требуется выбор субградиента или специальная процедура.

**Две разные квадратичные задачи:** исходный риск может быть log-loss, а дерево на очередном шаге обучают приближать числа $s_i$ по $\sum_i(s_i-h(x_i))^2$. Это способ подобрать направление в пространстве предсказаний, а не замена исходной цели. Затем шаг или прогнозы листьев можно подбирать по исходной потере.

In [ ]:
import numpy as np

y = np.array([10., 20., 30.])
pred = np.array([12., 15., 25.])
s = y - pred  # MSE: антиградиент без общего множителя 1/n
print("Текущий прогноз:", pred)
print("Псевдоостатки:", s)
print("Прогноз после точного шага:", pred + s)

## 4. Алгоритм и пример на Python

1. Выбрать $F_0$ (для MSE — среднее ответов).
2. Посчитать $s_i$ по **исходной** функции потерь при текущих прогнозах.
3. Обучить небольшое регрессионное дерево $h_m(x_i)\approx s_i$.
4. При необходимости найти $\gamma_m=\arg\min_\gamma\sum_i L(y_i,F_{m-1}(x_i)+\gamma h_m(x_i))$ или оптимизировать значения листьев.
5. Обновить $F_m=F_{m-1}+\eta\gamma_m h_m$, повторить и оценить качество на отложенной выборке.

Следующий код реализует только простой вариант для MSE с фиксированным $\eta$. Дерево обучается на остатках; это помогает увидеть сумму моделей без деталей промышленных библиотек.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(42)
X = np.linspace(-3, 3, 160).reshape(-1, 1)
y = np.sin(2 * X[:, 0]) + 0.2 * X[:, 0] + rng.normal(0, 0.16, len(X))
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

initial = float(y_train.mean())
train_pred = np.full(len(X_train), initial)
val_pred = np.full(len(X_val), initial)
eta = 0.15
trees = []
history = []
for step in range(30):
    residual = y_train - train_pred
    tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=5, random_state=step)
    tree.fit(X_train, residual)
    trees.append(tree)
    train_pred += eta * tree.predict(X_train)
    val_pred += eta * tree.predict(X_val)
    history.append((mean_squared_error(y_train, train_pred),
                    mean_squared_error(y_val, val_pred)))

print(f"F0 = {initial:.3f}; после 30 деревьев: train MSE = {history[-1][0]:.3f}, validation MSE = {history[-1][1]:.3f}")

grid = np.linspace(-3.2, 3.2, 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_train[:, 0], y_train, s=12, alpha=.5, label="Обучение")
for count in [0, 1, 5, 30]:
    curve = initial + eta * sum((tree.predict(grid) for tree in trees[:count]), np.zeros(len(grid)))
    axes[0].plot(grid[:, 0], curve, label=f"{count} деревьев")
axes[0].legend(); axes[0].set_title("Последовательные поправки")
axes[1].plot(np.arange(1, 31), history)
axes[1].legend(["train MSE", "validation MSE"])
axes[1].set_xlabel("Число деревьев"); axes[1].set_title("Ошибка ансамбля")
plt.tight_layout(); plt.show()

## 5. Глубина деревьев и контроль переобучения

> **Короткий ответ. Почему фиксируют максимальную глубину?**  
> Глубина ограничивает сложность каждого отдельного шага бустинга: неглубокое дерево исправляет устойчивую часть ошибки, а не пытается сразу запомнить остатки и шум. Одинаковый предел глубины сохраняет регуляризацию постоянной на всех итерациях, упрощает подбор гиперпараметров и делает затраты предсказуемыми. Фиксируется именно **максимум**: дерево может остановиться раньше, если полезного разбиения нет. Это практический способ контроля сложности, а не обязательное математическое условие бустинга.

### Почему максимальную глубину обычно фиксируют заранее и делают одинаковой для всех деревьев

В записи бустинга $h_m\in\mathcal H$ все поправки выбираются из одного класса базовых моделей $\mathcal H$. Для деревьев этот класс удобно задать условием $\operatorname{depth}(h)\le D$. Поэтому на каждой итерации меняются разбиения и значения листьев, но ограничение сложности остаётся одним и тем же. **Фиксируется верхняя граница**, а не фактическая глубина: дерево может закончить рост раньше, если разбиение не улучшает критерий или нарушает `min_samples_leaf`.

Это полезно по четырём причинам:

1. **Контролируемая сложность шага.** Одно дерево должно дать небольшую устойчивую поправку к ансамблю, а не решить всю задачу за один шаг. Глубокое дерево точнее приближает псевдоостатки на обучении, но вместе с сигналом легче запоминает шум.
2. **Постоянная регуляризация.** Пара $(D,\eta)$ задаёт силу каждого шага: $D$ ограничивает форму поправки, а learning rate $\eta$ — её амплитуду. Если без отдельного правила увеличивать $D$, регуляризация незаметно ослабевает по ходу обучения.
3. **Понятный гиперпараметр.** Одно значение $D$ можно выбрать кросс-валидацией. Расписание $(D_1,\ldots,D_M)$ резко расширяет пространство поиска и повышает риск подстроить его под валидационную выборку.
4. **Предсказуемые ресурсы.** У бинарного дерева глубины не больше $D$ максимум $2^D$ листьев и путь прогноза длины не больше $D$. Это даёт управляемые время и память.

Есть и содержательная интерпретация: дерево глубины $D$ способно в одном пути совместно использовать не более $D$ условий, поэтому глубина задаёт допустимый порядок взаимодействий признаков. Пни (`max_depth=1`) моделируют сумму одномерных эффектов; глубина 2 уже позволяет парные взаимодействия. Это ориентир, а не строгая гарантия: разные деревья ансамбля вместе могут образовать более сложную функцию.

Почему не наращивать глубину, полагая, что остатки становятся сложнее? Остатки не обязаны становиться сложнее: по мере извлечения сигнала в них обычно падает отношение сигнал/шум. Позднее глубокое дерево поэтому особенно легко подгоняет случайные отклонения. Если простых деревьев недостаточно для устойчивого взаимодействия признаков, значение $D$ лучше выбрать по валидации для всего ансамбля, а не автоматически увеличивать по номеру итерации.

**Важно:** одинаковая максимальная глубина — не математическое требование алгоритма. Можно смешивать деревья разной сложности или задавать расписание глубины, но тогда это уже дополнительный гиперпараметр, который тоже нужно проверять на валидации. На практике вместе с глубиной регулируют `min_samples_leaf`, число листьев, $\eta$, число итераций и раннюю остановку.

In [ ]:
# Сравним одинаковую глубину с произвольным расписанием, где деревья становятся глубже.
def fit_with_depth_schedule(depths, learning_rate=0.10):
    train_hat = np.full(len(y_train), y_train.mean())
    val_hat = np.full(len(y_val), y_train.mean())
    scores = []
    for step, depth in enumerate(depths):
        tree = DecisionTreeRegressor(
            max_depth=depth, min_samples_leaf=5, random_state=step
        )
        tree.fit(X_train, y_train - train_hat)
        train_hat += learning_rate * tree.predict(X_train)
        val_hat += learning_rate * tree.predict(X_val)
        scores.append((mean_squared_error(y_train, train_hat),
                       mean_squared_error(y_val, val_hat)))
    return np.asarray(scores)

n_steps = 80
fixed_depths = [2] * n_steps
growing_depths = np.linspace(1, 6, n_steps).astype(int)
fixed_scores = fit_with_depth_schedule(fixed_depths)
growing_scores = fit_with_depth_schedule(growing_depths)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for scores, label in [(fixed_scores, 'фиксированная D=2'),
                      (growing_scores, 'D растёт от 1 до 6')]:
    axes[0].plot(scores[:, 0], label=label)
    axes[1].plot(scores[:, 1], label=label)
axes[0].set_title('Ошибка на обучении')
axes[1].set_title('Ошибка на валидации')
for ax in axes:
    ax.set_xlabel('Число деревьев')
    ax.set_ylabel('MSE')
    ax.legend()
plt.tight_layout()
plt.show()

for name, scores in [('Фиксированная D=2', fixed_scores),
                     ('Растущая D', growing_scores)]:
    best_step = int(np.argmin(scores[:, 1]))
    print(f'{name}: минимум validation MSE = {scores[best_step, 1]:.4f} '
          f'после {best_step + 1} деревьев; train MSE = {scores[-1, 0]:.4f}')

В этом примере растущая глубина должна быстрее уменьшать ошибку обучения, но это само по себе не доказывает лучшее обобщение: решение выбирают по минимуму валидационной ошибки. Один эксперимент не устанавливает универсального победителя — он показывает, почему расписание глубины нельзя обосновать только улучшением на train. Для корректной оценки ранней остановки используют валидацию, а не тестовую выборку.

Итак, фиксированная глубина — это простой способ сохранить одинаковый бюджет сложности каждого градиентного шага. Это **не правило «всегда глубина 2»**: достаточная глубина зависит от взаимодействий в данных, размера выборки, ограничений листьев и остальных видов регуляризации.

## 6. Бустинг и случайный лес: сложность

Обозначим $n$ — число объектов, $d$ — число проверяемых признаков, $D$ — максимальная глубина дерева, $M$ — число деревьев. В идеализированной схеме после подготовки кандидатов поиск лучшего порога по одному признаку в узле занимает $O(n_{узла})$: статистики обновляют за один проход. На одном уровне сумма объектов по узлам не превышает $n$, поэтому стоимость дерева $O(Ddn)$.

| Операция | Бустинг | Случайный лес |
|---|---:|---:|
| Обучение, последовательная сумма работы | $O(MDdn)$ | $O(MDfn)$, где $f\le d$ признаков пробуют в узле |
| Прогноз одного объекта | $O(MD)$ | $O(MD)$ |
| Память на деревья | $O(M2^D)$ в грубой верхней оценке | $O(M2^D)$ |

Для леса bootstrap обычно берёт порядка $n$ строк на дерево; при $f=d$ и одинаковых $M,D$ **оценки общего объёма работы одного порядка**. Фактические $D$, $M$, $f$ и стоимость разбиения часто различаются. Лес может строить деревья параллельно, а бустинг зависит от предыдущих деревьев. Поэтому одинаковый Big O не означает одинаковое время работы.

**Условия оценки:** сортировка непрерывных значений или построение гистограмм добавляет собственную стоимость (например, первоначальная сортировка порядка $O(dn\log n)$ в простой схеме). Наивный пересчёт всех порогов и некоторые реализации дают другие оценки. Память $O(M2^D)$ — грубый верхний предел числа узлов, а не типичный размер модели.

## 7. XGBoost, LightGBM, CatBoost

Три популярные библиотеки реализуют деревья с градиентным бустингом, но используют разные инженерные решения. `scikit-learn` (`GradientBoostingRegressor`, `HistGradientBoostingRegressor`) удобен как учебная основа и базовая модель.

| Библиотека | Основной акцент | Практический ориентир |
|---|---|---|
| **XGBoost** | Регуляризация дерева и весов листьев, градиенты и вторые производные; гистограммный поиск разбиений (`hist`) | Гибкие настройки, в том числе регуляризация и работа с пропусками; начать с ограниченной глубины |
| **LightGBM** | Гистограммы и рост по листу с наибольшим улучшением (`leaf-wise`) | Удобен для больших табличных данных; на малой выборке следить за `num_leaves` и размером листа |
| **CatBoost** | Специальная обработка категориальных признаков и упорядоченные методы против смещения при использовании целевой статистики | Удобен при наличии категориальных столбцов; указывать их явно, следить за проверкой без утечки |

Ни одна из библиотек не является лучшей для всех наборов данных. Сравнивать следует на одинаковых разбиениях и метрике. 

## 8. Как выбрать библиотеку и как устроены её деревья

> **Короткий ответ. Как устроены деревья в трёх библиотеках?**  
> **XGBoost** обычно растит обычные несимметричные деревья по уровням (`depthwise`): разные узлы одного уровня могут использовать разные условия. **LightGBM** растит обычное дерево по лучшему листу (`leaf-wise`), поэтому одна ветвь может стать намного глубже остальных. **CatBoost** по умолчанию строит симметричное (`oblivious`) дерево: на каждом уровне одно условие применяется ко всем узлам, поэтому все пути имеют одинаковую длину. У XGBoost и CatBoost есть и альтернативные политики роста.

### 8.1. Сначала — не название библиотеки, а свойства данных

Выбор ниже — практическая стартовая точка, а не теорема. Окончательное решение принимают по одной и той же схеме валидации, метрике и бюджету вычислений. Нельзя честно объявить библиотеку победителем, обучив XGBoost на одном сплите, а LightGBM — на другом.

| Данные и ограничения | С чего разумно начать | Почему | Что проверить |
|---|---|---|---|
| Много категориальных признаков, высокая кардинальность, хочется минимум ручного кодирования | **CatBoost** | Нативно строит статистические числовые признаки из категорий и их комбинаций; ordered-подход уменьшает утечку target statistics | Передать исходные категории через `cat_features`, а не заранее one-hot кодировать всё |
| Широкая или большая табличная выборка, важны скорость CPU и память | **LightGBM** | Гистограммы и leaf-wise рост быстро находят сильные локальные разбиения; есть нативные категории | На малых данных leaf-wise рост легко переобучается: ограничивать `num_leaves`, `max_depth`, `min_child_samples` |
| В основном числовые/уже закодированные признаки, нужен универсальный и хорошо регуляризуемый baseline | **XGBoost** | Зрелый общий tree booster: `hist`, L1/L2 для значений листьев, `gamma`, sampling, ограничения взаимодействий и монотонности | Начать с `tree_method='hist'`; совместно настраивать глубину, `min_child_weight`, sampling и learning rate |
| Очень маленькая простая выборка | Сначала линейная модель или обычный sklearn-бустинг; затем **CatBoost/XGBoost** как сравнение | Сложный бустинг не гарантирует выигрыш, а оценка качества имеет большой разброс | Repeated/nested CV, простой baseline, доверительные интервалы |
| GPU и большой датасет | Все три — кандидаты | Все поддерживают GPU, но допустимые параметры и алгоритмы различаются | Измерять wall time, память и качество именно на доступном железе |
| Ranking, survival, специальные функции потерь или ограничения | Проверить поддержку objective в каждой библиотеке | Здесь тип задачи важнее общей репутации библиотеки | Сверить objective, метрику, CPU/GPU-ограничения и формат данных |

**Категориальные признаки — важная оговорка.** CatBoost раскрывает своё главное преимущество, когда получает категории нативно. Если перед ним поставить общий `OneHotEncoder`, модель остаётся CatBoost, но сравнение уже не демонстрирует его ordered categorical statistics. LightGBM умеет искать разбиение множества категорий на две группы, если признак передан как categorical. XGBoost также поддерживает категории с one-hot либо partition-based splits, однако это нужно явно включить и использовать совместимый tree method.

**Пропуски.** Все три библиотеки умеют работать с пропущенными числовыми значениями. Для каждого разбиения алгоритм может выучить направление по умолчанию для объектов с пропуском; конкретные режимы и ограничения зависят от библиотеки. Поэтому предварительная импутация не всегда обязательна, но её эффект всё равно нужно проверять по CV.

### 8.2. XGBoost: обычные несимметричные деревья с несколькими стратегиями роста

XGBoost использует **не oblivious-деревья**: разные узлы одного уровня могут выбрать разные признаки и пороги. Поэтому дерево в общем случае несимметрично по правилам разбиения и может иметь ветви разной фактической длины.

По умолчанию `grow_policy='depthwise'`: сначала рассматриваются узлы, ближайшие к корню. Это рост по уровням, но **не симметричность** — одинаковый split во всех узлах уровня не требуется. При `grow_policy='lossguide'` выбирается лист с максимальным улучшением loss, поэтому форма обычно ещё более несбалансированная. `max_depth` ограничивает длину пути, а для `lossguide` особенно естественен `max_leaves`.

Способ поиска порога задаёт `tree_method`:

- `exact` перебирает кандидаты по исходным значениям и хуже масштабируется;
- `approx` использует приближённые квантили и гистограммы;
- `hist` один раз квантует значения в bins и строит гистограммы — это современная практическая отправная точка и значение, к которому сводится `auto`.

Кандидат оценивается через градиенты и вторые производные loss. Структуру контролируют `max_depth`/`max_leaves`, минимальная сумма гессианов в потомке `min_child_weight` и минимальный выигрыш `gamma`. Значения листьев дополнительно регуляризуются через `reg_lambda` (L2) и `reg_alpha` (L1). Строки и признаки можно подвыбирать через `subsample` и семейство `colsample_by*`. Иными словами, XGBoost регулирует отдельно **форму дерева**, **допуск разбиения** и **величину ответа листа**.

Схематично depth-wise дерево может выглядеть так (условия в узлах одного уровня различаются):

```text
                 x1 < 4.2
               /          \
          x3 < 0.7       x2 < 11
          /     \             \
       leaf    leaf          x5 < 2
                              /   \
                           leaf  leaf
```

Официальные источники: [Tree Methods](https://xgboost.readthedocs.io/en/stable/treemethod.html), [XGBoost Parameters](https://xgboost.readthedocs.io/en/stable/parameter.html).

### 8.3. LightGBM: leaf-wise, обычно глубокие и асимметричные деревья

LightGBM по умолчанию растит дерево **leaf-wise (best-first)**: среди всех текущих листьев выбирается тот, чьё разбиение сильнее всего уменьшает loss. Соседний лист может больше никогда не разделиться. Поэтому дерево обычно несимметричное и несбалансированное: одна ветвь бывает заметно глубже другой.

Такой рост часто уменьшает training loss быстрее, чем level-wise рост при одинаковом числе листьев, но на малой выборке легче подгоняет шум. Главный параметр сложности — `num_leaves`, а не только `max_depth`. Ограничение `max_depth` не меняет leaf-wise алгоритм, а лишь запрещает слишком длинные пути. `min_data_in_leaf`/`min_child_samples` и `min_sum_hessian_in_leaf` не дают создавать слабоподдержанные листья; `min_gain_to_split` требует минимального улучшения. Для дерева глубины $D$ грубый полный предел равен $2^D$ листьев, но у leaf-wise дерева обычно разумно брать `num_leaves` существенно меньше этого предела.

Числовые значения квантуются в `max_bin` интервалов, и gain считается по агрегатам градиентов/гессианов в гистограмме. Это экономит время и память. LightGBM также использует вычитание гистограмм: зная гистограмму родителя и одного потомка, получает гистограмму второго. Дополнительные оптимизации — GOSS для отбора объектов по градиентам и EFB для объединения редко одновременно ненулевых признаков.

Схематично best-first рост выглядит так:

```text
                 x1 < 4.2
               /          \
            leaf         x2 < 11       <- разделился лучший лист
                        /       \
                     leaf      x5 < 2  <- снова правая ветвь
                              /    \
                           leaf   leaf
```

Для категориального признака LightGBM может искать оптимальное разбиение категорий на две группы вместо one-hot разбиения каждой категории. Это особенно полезно при средней и высокой кардинальности, но требует корректно указать categorical features.

Официальные источники: [Features](https://lightgbm.readthedocs.io/en/stable/Features.html), [Parameters Tuning](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html), [Advanced Topics](https://lightgbm.readthedocs.io/en/stable/Advanced-Topics.html).

### 8.4. CatBoost: симметричные oblivious-деревья по умолчанию

Стандартный `grow_policy='SymmetricTree'` строит дерево уровень за уровнем и на каждом уровне применяет **одно и то же условие ко всем листьям текущего уровня**. Поэтому структура симметрична: при глубине $D$ полное дерево имеет $2^D$ листьев, а каждый объект проходит ровно $D$ бинарных проверок. Такие деревья называют symmetric или oblivious. Они ограничивают множество возможных структур, зато быстро применяются, компактно представляются и действуют как дополнительная регуляризация.

```text
уровень 1:                 x1 < 4.2
                           /       \
уровень 2:           x3 < 0.7     x3 < 0.7   <- то же условие
                     /   \         /   \
уровень 3:        x2<11 x2<11    x2<11 x2<11 <- то же условие
```

Однако «CatBoost всегда симметричен» — неверно. Доступны ещё две политики:

- `Depthwise`: рост по уровням, но каждый лист выбирает собственный лучший split;
- `Lossguide`: разделяется лист с лучшим улучшением, пока не достигнут `max_leaves`.

Для стандартных симметричных деревьев основной структурный параметр — `depth`; `min_data_in_leaf` и `max_leaves` относятся только к `Depthwise`/`Lossguide`. Значения листьев могут оцениваться несколькими градиентными или ньютоновскими шагами (`leaf_estimation_iterations`).

Главное отличие CatBoost выходит за рамки формы дерева. До выбора split категории преобразуются в числовые статистики и их комбинации. Ordered statistics используют только доступную предыдущую часть перестановки, чтобы значение признака для объекта не вычислялось напрямую из его собственного target. Ordered boosting аналогично уменьшает prediction shift; точный default режима зависит от устройства, размера данных и задачи. Это особенно ценно на небольших данных с категориями, но требует передавать исходные категориальные столбцы через `cat_features`.

Официальные источники: [Tree growing policy](https://catboost.ai/docs/en/concepts/parameter-tuning), [Choosing the tree structure](https://catboost.ai/docs/en/concepts/algorithm-main-stages_choose-tree-structure), [Categorical feature transformation](https://catboost.ai/docs/en/concepts/algorithm-main-stages_cat-to-numberic).

### 8.5. Сводная таблица характеристик деревьев

| Характеристика | XGBoost | LightGBM | CatBoost по умолчанию |
|---|---|---|---|
| Базовая структура | Обычное бинарное дерево | Обычное бинарное дерево | Oblivious / symmetric tree |
| Симметричность | Нет | Нет | Да: один split на весь уровень |
| Стратегия роста по умолчанию | `depthwise` | `leaf-wise` / best-first | `SymmetricTree`, level-wise |
| Альтернативный рост | `lossguide` | Основной алгоритм остаётся leaf-wise | `Depthwise`, `Lossguide` |
| Главный контроль размера | `max_depth`; при lossguide также `max_leaves` | `num_leaves` + `max_depth` | `depth`; при Lossguide — `max_leaves` |
| Поиск порогов | `exact`, `approx`, обычно `hist` | Гистограммы | Предварительно рассчитанные границы / бинаризованные признаки |
| Вторые производные | Используются в gain и весах листьев | Используются в gain и весах листьев | Поддерживает gradient/Newton leaf estimation и score-функции |
| Категории | Нативная one-hot/partition поддержка при совместимом tree method | Нативное разбиение категорий на группы | Ordered target statistics и комбинации категорий |
| Типичный риск | Глубокие деревья и слабая регуляризация | Leaf-wise переобучение на малых данных | Экспоненциальный рост $2^D$ листьев с глубиной |
| Сильная сторона | Гибкость и явная регуляризация | Скорость на больших широких таблицах | Категории, малые/средние таблицы, быстрый inference симметричных деревьев |

### Практический порядок выбора

1. Построить простой baseline и зафиксировать CV-схему и метрику.
2. Если много исходных категорий — первой проверить CatBoost **без общего one-hot preprocessing**.
3. Если таблица большая/широкая и важна скорость — первой проверить LightGBM с консервативными `num_leaves` и `min_child_samples`.
4. Если признаки числовые или уже закодированы и нужен универсальный baseline — первой проверить XGBoost с `tree_method='hist'`.
5. Затем сравнить минимум две библиотеки на одних фолдах: выбор по названию не заменяет эксперимент. Учитывать не только среднюю метрику, но и разброс по фолдам, время, память, калибровку и стоимость inference.